# 🏋️ Exercises Dataset - 1-Click Transparent Background Batch Processor
This notebook uses a free **Nvidia T4 Cloud GPU** to remove the background from all 1,324 exercise images and animated GIFs using **BiRefNet / RMBG-2.0**.

### Why this is the best method:
- **Preserves internal whites**: White shoes, socks, bar highlights, and muscle shading are 100% kept.
- **Smooth 8-bit Alpha**: Creates transparent WebP / APNG / GIF files ready for mobile apps (dark mode / light mode) without white halos.
- **Speed**: Runs on GPU, processing the entire 1,324 dataset in ~15 minutes instead of hours.

In [ ]:
# 1. Install GPU-accelerated rembg and dependencies
!pip install -q rembg[gpu] onnxruntime-gpu pillow tqdm

In [ ]:
# 2. Upload or clone your repository
# If using GitHub / Drive:
# !git clone https://github.com/your-username/exercises-dataset.git
# %cd exercises-dataset

# Or upload your images/ and videos/ folders into Colab directly

In [ ]:
import os
from pathlib import Path
from PIL import Image, ImageSequence
from rembg import new_session, remove
from tqdm import tqdm

# Initialize BiRefNet (State-of-the-art segmentation)
session = new_session('birefnet-general')

images_in = Path('./images')
images_out = Path('./transparent_images')
videos_in = Path('./videos')
videos_out = Path('./transparent_videos')

images_out.mkdir(parents=True, exist_ok=True)
videos_out.mkdir(parents=True, exist_ok=True)

# Process Static Thumbnails
print('Processing static thumbnails...')
jpg_files = list(images_in.glob('*.jpg'))
for f in tqdm(jpg_files, desc='Thumbnails'):
    target = images_out / f'{f.stem}.png'
    if not target.exists():
        with Image.open(f) as img:
            out = remove(img.convert('RGBA'), session=session)
            out.save(target, 'PNG')

# Process Animated GIFs
print('\nProcessing animation GIFs...')
gif_files = list(videos_in.glob('*.gif'))
for f in tqdm(gif_files, desc='Animations'):
    target_webp = videos_out / f'{f.stem}.webp'
    target_gif = videos_out / f'{f.stem}.gif'
    if not target_webp.exists():
        with Image.open(f) as im:
            frames = []
            durations = []
            for frame in ImageSequence.Iterator(im):
                durations.append(frame.info.get('duration', 100))
                clean_frame = remove(frame.convert('RGBA'), session=session)
                frames.append(clean_frame)
            # Save high-fidelity Animated WebP with 8-bit alpha
            frames[0].save(target_webp, save_all=True, append_images=frames[1:], duration=durations, loop=0, format='WEBP', lossless=True)
            # Also save transparent GIF
            frames[0].save(target_gif, save_all=True, append_images=frames[1:], duration=durations, loop=0, format='GIF', transparency=0, disposal=2)

print('\nDone! Creating zip download...')
!zip -q -r transparent_assets.zip transparent_images transparent_videos
print('Download transparent_assets.zip ready!')